# C3-VCell synthetic quick start

Run from the repository root in an environment installed with `python -m pip install -e '.[test]'`. This artificial example checks the interface, not biological performance. See [the full tutorial](../docs/quickstart.zh-CN.md).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'examples' and (root.parent / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from the repository root or examples directory')
os.chdir(root)
work = Path(os.environ.get('C3_TUTORIAL_RUN_DIR', 'runs/notebook-toy'))
work.mkdir(parents=True, exist_ok=False)

def run_cli(*args):
    subprocess.run([sys.executable, '-m', 'c3vcell.cli', *map(str, args)], check=True)

subprocess.run([sys.executable, 'examples/make_toy_data.py', '--output', str(work / 'raw.h5ad')], check=True)

## Split and train

G4 and G5 are held out. The few training steps are only a software smoke test.

In [ ]:
run_cli('prepare', '--input', work / 'raw.h5ad', '--outdir', work / 'split',
        '--max-targets', 0, '--max-per-target', 40, '--max-controls', 120,
        '--max-genes', 64, '--holdout-targets', 'G4', 'G5', '--seed', 42)
run_cli('train', '--train', work / 'split/train.h5ad', '--outdir', work / 'model',
        '--latent-dim', 8, '--hidden-dim', 16, '--pretrain-steps', 5,
        '--flow-steps', 5, '--batch-size', 16, '--seed', 42)

## Predict with and without a reference expression file

The prediction inputs, targets and seed are matched. The chosen strength is not calibrated.

In [ ]:
common = ['--cells-per-target', 20, '--steps', 2, '--sampling', 'residual-rounding',
          '--effect-strength', 1, '--seed', 42]
run_cli('predict', '--checkpoint', work / 'model/model.pt',
        '--reference', work / 'split/reference.h5ad', '--controls', work / 'split/input_controls.h5ad',
        '--output', work / 'prediction.h5ad', '--control-sampling', 'input-controls', *common)
run_cli('predict-targets', '--checkpoint', work / 'model/model.pt',
        '--targets', 'examples/toy_targets.csv', '--controls', work / 'split/input_controls.h5ad',
        '--output', work / 'requested.h5ad', *common)
run_cli('control-baseline', '--reference', work / 'split/reference.h5ad',
        '--controls', work / 'split/input_controls.h5ad', '--output', work / 'control.h5ad',
        '--cells-per-target', 20, '--control-sampling', 'input-controls', '--seed', 42)

## Local diagnostics

One VCC preset raw metric and one unscored diagnostic are computed. They are not an official leaderboard score.

In [ ]:
for name in ['prediction', 'control']:
    run_cli('score', '--prediction', work / f'{name}.h5ad',
            '--reference', work / 'split/reference.h5ad', '--outdir', work / f'{name}-score',
            '--metrics', 'pds_cosine', 'expr_mse_unbiased_capped', '--threads', 2)

import anndata as ad
import scipy.sparse as sp
prediction = ad.read_h5ad(work / 'prediction.h5ad')
requested = ad.read_h5ad(work / 'requested.h5ad')
assert prediction.shape == requested.shape == (40, 64)
assert prediction.obs['source_control_id'].tolist() == requested.obs['source_control_id'].tolist()
assert (sp.csr_matrix(prediction.X) != sp.csr_matrix(requested.X)).nnz == 0
print('Matching synthetic predictions:', prediction.shape)
print('Scoring provenance:', work / 'prediction-score/scoring-provenance.json')

## Synthetic epigenomic prior

These GTF, peak, Hi-C and methylation inputs are deliberately invented. Their GRCh38 label exists only to exercise schema checks and does not make the coordinates biological.

In [ ]:
subprocess.run([sys.executable, 'examples/make_toy_epigenome.py', '--source',
                str(work / 'raw.h5ad'), '--outdir', str(work / 'epi-inputs')], check=True)
run_cli('build-epigenetic-bundle', '--gene-data', work / 'split/train.h5ad',
        '--spec', work / 'epi-inputs/sources.json', '--outdir', work / 'epigenome')
run_cli('shuffle-epigenetic-bundle', '--bundle', work / 'epigenome',
        '--gene-data', work / 'split/train.h5ad', '--outdir', work / 'graph-shuffle',
        '--graph-only', '--seed', 42)
run_cli('train', '--train', work / 'split/train.h5ad', '--outdir', work / 'epi-model',
        '--epigenetic-bundle', work / 'epigenome', '--latent-dim', 8, '--hidden-dim', 16,
        '--pretrain-steps', 5, '--flow-steps', 5, '--batch-size', 16, '--seed', 42)
print('Synthetic graph and model:', work / 'epigenome', work / 'epi-model/model.pt')